# KnottedGraph vs Topoly: long-range Yamada scaling

This benchmark compares **identical planar-diagram inputs** over substantially longer ranges than the earlier short benchmark.

Correctness is enforced before a timing pair is accepted:

\[
P_{\mathrm{Topoly}}(A)=\pm A^k P_{\mathrm{KG}}(A^{\pm1}).
\]

The benchmark contains four complementary families:

- projected-crossing scaling, targeted through $c=30$;
- edge scaling through $E=200$ on a two-vertex multiedge family;
- vertex/input-size scaling through $V=512$ on cycles;
- a connected trivalent prism family targeted through $V=60, E=90$.

Difficult families use a hard per-framework timeout. A timeout is plotted as a censored point at the timeout threshold instead of allowing the benchmark to hang.

The fitted power-law/exponential curves below are **empirical scaling summaries**, not formal complexity proofs.


In [ ]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "knotted_graph").exists():
    raise RuntimeError("Run this notebook from inside the KnottedGraph checkout.")

SRC = ROOT / "src"
sys.path.insert(0, str(SRC))

import knotted_graph
kg_path = Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents:
    raise RuntimeError(f"A stale knotted_graph was imported from {kg_path}")

try:
    import topoly
except ImportError as exc:
    raise ImportError("Install Topoly first: pip install topoly") from exc

OUT = ROOT / "User_guide" / "benchmarks"
RES = OUT / "results_latest"
FIG = OUT / "figures_latest"
RES.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("KnottedGraph:", kg_path)
print("Topoly:", Path(topoly.__file__).resolve())


## 1. Run the extended correctness-gated benchmark

Each framework/case is executed in a separate subprocess. The default hard limit is 10 seconds per framework per case. This is long enough to establish a useful curve while making the transition into an exponential regime explicit.


In [ ]:
script = ROOT / "dev" / "benchmark_topoly_extended_scaling.py"
env = dict(os.environ)
env["PYTHONPATH"] = str(SRC)
env["PYTHONNOUSERSITE"] = "1"

proc = subprocess.run(
    [sys.executable, str(script), "--timeout", "10"],
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
    timeout=1800,
)
print(proc.stdout)
if proc.returncode:
    raise RuntimeError(
        f"Extended Topoly benchmark failed with exit code {proc.returncode}.\n"
        f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
    )

for line in reversed(proc.stdout.splitlines()):
    if line.startswith("SUMMARY="):
        rows = json.loads(line[8:])
        break
else:
    raise RuntimeError("Benchmark completed without SUMMARY= output.")

print("rows:", len(rows))


In [ ]:
keys = list(dict.fromkeys(key for row in rows for key in row))
with (RES / "topoly_extended_scaling.csv").open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=keys)
    writer.writeheader()
    writer.writerows(rows)

families = {}
for row in rows:
    families.setdefault(row["family"], []).append(row)

for family, family_rows in families.items():
    ok = sum(
        row["knottedgraph_status"] == "ok" and row["topoly_status"] == "ok"
        for row in family_rows
    )
    print(f"{family:18s}: {ok}/{len(family_rows)} paired results")


## 2. Helpers for censored plots and empirical fits

For successful points, the measured median runtime is plotted. If a framework reaches the 10-second hard limit, the point is shown at 10 seconds using an open marker.

For uncensored positive points we fit

\[t(x)=C x^\alpha\]

and

\[t(x)=C e^{\beta x}.\]

The notebook reports $R^2$ in log-time. These fits help identify which empirical model better describes the sampled range; they do **not** prove an asymptotic Big-$O$ theorem.


In [ ]:
def values_for(rows_, xkey, framework):
    xs, ys, censored = [], [], []
    status_key = f"{framework}_status"
    time_key = f"{framework}_s"
    for row in sorted(rows_, key=lambda r: r[xkey]):
        if xkey not in row:
            continue
        xs.append(float(row[xkey]))
        if row[status_key] == "ok":
            ys.append(float(row[time_key]))
            censored.append(False)
        elif row[status_key] == "timeout":
            ys.append(float(row["timeout_s"]))
            censored.append(True)
        else:
            ys.append(np.nan)
            censored.append(True)
    return np.asarray(xs), np.asarray(ys), np.asarray(censored)

def fit_models(xs, ys, censored):
    mask = (~censored) & np.isfinite(ys) & (ys > 0) & (xs > 0)
    x = xs[mask]
    y = ys[mask]
    if len(x) < 3:
        return {}
    logy = np.log(y)
    px = np.log(x)
    alpha, logC_power = np.polyfit(px, logy, 1)
    pred_power = logC_power + alpha * px
    beta, logC_exp = np.polyfit(x, logy, 1)
    pred_exp = logC_exp + beta * x

    def r2(actual, predicted):
        ss_res = np.sum((actual - predicted) ** 2)
        ss_tot = np.sum((actual - actual.mean()) ** 2)
        return 1.0 - ss_res / ss_tot if ss_tot else 1.0

    return {
        "power_alpha": float(alpha),
        "power_C": float(np.exp(logC_power)),
        "power_r2": float(r2(logy, pred_power)),
        "exp_beta": float(beta),
        "exp_C": float(np.exp(logC_exp)),
        "exp_r2": float(r2(logy, pred_exp)),
    }

def plot_family(rows_, xkey, xlabel, title, stem):
    plt.figure(figsize=(9, 5.8))
    fit_report = {}
    for framework, label, marker in [
        ("knottedgraph", "KnottedGraph", "o"),
        ("topoly", "Topoly", "s"),
    ]:
        xs, ys, censored = values_for(rows_, xkey, framework)
        measured = ~censored & np.isfinite(ys)
        timed_out = censored & np.isfinite(ys)
        plt.plot(xs[measured], ys[measured], marker=marker, label=label)
        if timed_out.any():
            plt.scatter(xs[timed_out], ys[timed_out], marker=marker, facecolors="none", label=f"{label} timeout/censored")
        fits = fit_models(xs, ys, censored)
        fit_report[framework] = fits
        if fits and measured.any():
            xx = np.geomspace(xs[measured].min(), xs[measured].max(), 200)
            power = fits["power_C"] * xx ** fits["power_alpha"]
            plt.plot(xx, power, linestyle="--", label=f"{label} power fit: alpha={fits['power_alpha']:.2f}, R2={fits['power_r2']:.3f}")
    plt.yscale("log")
    plt.xscale("log" if xkey in {"V", "E"} else "linear")
    plt.xlabel(xlabel)
    plt.ylabel("Yamada evaluation time (s)")
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG / f"{stem}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIG / f"{stem}.pdf", bbox_inches="tight")
    plt.show()
    for framework, fits in fit_report.items():
        if fits:
            print(f"{framework}: power alpha={fits['power_alpha']:.4g}, R2={fits['power_r2']:.4f}; exp beta={fits['exp_beta']:.4g}, R2={fits['exp_r2']:.4f}")


## 3. Crossing-number scaling

The controlled crossing family keeps the individual components small while increasing the exact number of projected crossings. This isolates the crossing-state contribution and extends the requested range toward $c=30$.


In [ ]:
plot_family(
    families["crossings"],
    "crossings",
    "Projected crossings, c",
    "Identical-PD Yamada scaling with projected crossing number",
    "topoly_vs_knottedgraph_crossings_long_range",
)


## 4. Edge-number scaling to $E=200$

The multiedge theta family fixes $V=2$, has no projected crossings, and increases only the number of graph edges. This provides a clean long-range edge-complexity experiment without crossing-state growth confounding the horizontal axis.


In [ ]:
plot_family(
    families["edges_theta"],
    "E",
    "Graph edges, E",
    "Yamada scaling with edge count at fixed V=2 and c=0",
    "topoly_vs_knottedgraph_edges_long_range",
)


## 5. Vertex/input-size scaling to $V=512$

Cycles have $E=V$ and no crossings. Because their Yamada polynomial has a closed structural form, this family probes parser/input/graph-size overhead over hundreds of vertices without exponential state growth.


In [ ]:
plot_family(
    families["vertices_cycle"],
    "V",
    "Graph vertices, V",
    "Yamada scaling with vertex count on crossing-free cycles",
    "topoly_vs_knottedgraph_vertices_long_range",
)


## 6. Harder connected trivalent scaling

Circular ladders (prism graphs) are connected trivalent graphs with $V=2n$ and $E=3n$. This family is substantially less shortcut-dominated than cycles and the two-vertex theta family. The target extends to $n=30$, i.e. $V=60, E=90$, but the benchmark deliberately records a timeout boundary instead of waiting indefinitely once exact recurrence growth becomes too costly.


In [ ]:
prism = families["connected_prism"]
plot_family(prism, "V", "Graph vertices, V", "Connected trivalent prism scaling with V", "topoly_vs_knottedgraph_connected_prism_vertices")
plot_family(prism, "E", "Graph edges, E", "Connected trivalent prism scaling with E", "topoly_vs_knottedgraph_connected_prism_edges")


## 7. Paired speed ratios and correctness status

Only rows where both implementations completed are assigned a speed ratio. Every such row has already passed the exact Laurent-equivalence gate. Timeouts are reported separately and are never converted into fabricated timing ratios.


In [ ]:
for family in ("crossings", "edges_theta", "vertices_cycle", "connected_prism"):
    print(f"\n[{family}]")
    for row in families[family]:
        x = row["crossings"] if family == "crossings" else (row["E"] if family == "edges_theta" else row["V"])
        if row["correctness"] == "PASS":
            print(f"x={x:>4}  KG={row['knottedgraph_s']:.6g}s  Topoly={row['topoly_s']:.6g}s  Topoly/KG={row['topoly_over_kg']:.4g}x  PASS")
        else:
            print(f"x={x:>4}  KG={row['knottedgraph_status']}  Topoly={row['topoly_status']}")

assert all(row["correctness"] == "PASS" for row in rows if row["knottedgraph_status"] == "ok" and row["topoly_status"] == "ok")
print("\nPASS: every paired displayed timing passed polynomial equivalence.")
